# `CPPStructurePlot.map_structure`

Paint per-residue **CPP / CPP-SHAP feature impact** onto an interactive **3D protein structure**. Each feature in a `df_feat` is mapped to the residues it spans and its signed impact is painted residue-by-residue onto the protein cartoon, rendered with `py3Dmol`. A red-white-blue ramp shows where features raise (red) or lower (blue) the prediction; a `plddt` mode shows AlphaFold confidence instead. The view is interactive — rotate/zoom it, and export it with `write_html`.

This is a `pro` feature (needs `biopython` + `py3Dmol`).

In [1]:
import pandas as pd
import aaanalysis as aa
import aaanalysis.utils as ut

aa.options["verbose"] = False

## A `df_feat` and a real structure

We use a small real AlphaFold model — human lysozyme C (UniProt `P61626`) — fetched automatically by passing `uniprot=`. (You can equally pass `pdb='path/to/model.pdb'`.) The `df_feat` here carries a few illustrative features with a signed `feat_impact`; in practice it comes from `CPP.run` + `ShapModel`.

In [2]:
df_cat = aa.load_scales(name='scales_cat').head(5).reset_index(drop=True)
splits = ['Segment(1,2)', 'Segment(2,2)', 'Segment(1,1)', 'Pattern(C,1)', 'Segment(1,4)']
parts = ['TMD', 'TMD', 'JMD_N', 'TMD', 'JMD_C']
df_feat = pd.DataFrame({
    ut.COL_FEATURE: [f"{parts[i]}-{splits[i]}-{r[ut.COL_SCALE_ID]}" for i, r in df_cat.iterrows()],
    'category': df_cat[ut.COL_CAT], 'subcategory': df_cat[ut.COL_SUBCAT],
    'scale_name': df_cat[ut.COL_SCALE_NAME],
    'abs_auc': [0.2, 0.15, 0.3, 0.1, 0.25], 'abs_mean_dif': [0.3, 0.2, 0.5, 0.4, 0.35],
    'mean_dif': [0.3, -0.2, 0.5, -0.4, 0.25], 'std_test': 0.1, 'std_ref': 0.1,
    'feat_impact': [0.8, -0.5, 1.2, -0.3, 0.6]})
aa.display_df(df_feat, n_rows=10, show_shape=True)

DataFrame shape: (5, 10)


,feature,category,subcategory,scale_name,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,feat_impact
1,"TMD-Segment(1,2)-LINS030110",ASA/Volume,Accessible surface area (ASA),ASA (folded coil/turn),0.200000,0.300000,0.300000,0.100000,0.100000,0.800000
2,"TMD-Segment(2,2)-LINS030113",ASA/Volume,Accessible surface area (ASA),ASA (folded coil/turn),0.150000,0.200000,-0.200000,0.100000,0.100000,-0.500000
3,"JMD_N-Segment(1,1)-JANJ780101",ASA/Volume,Accessible surface area (ASA),ASA (folded protein),0.300000,0.500000,0.500000,0.100000,0.100000,1.200000
4,"TMD-Pattern(C,1)-JANJ780103",ASA/Volume,Accessible surface area (ASA),ASA (folded protein),0.100000,0.400000,-0.400000,0.100000,0.100000,-0.300000
5,"JMD_C-Segment(1,4)-LINS030104",ASA/Volume,Accessible surface area (ASA),ASA (folded protein),0.250000,0.350000,0.250000,0.100000,0.100000,0.600000


## Impact mode with fade-context

`mode="impact"` paints the white→red / white→blue ramp; `focus="fade"` ghosts residues outside the feature window; `size_by_impact` scales each residue's stick by `|impact|`. `start` is the absolute residue number of the first JMD-N residue (here the window covers residues 40–69).

In [3]:
cpps_plot = aa.CPPStructurePlot(jmd_n_len=10, jmd_c_len=10, verbose=False)
view = cpps_plot.map_structure(df_feat=df_feat, uniprot='P61626', col_imp='feat_impact',
                         tmd_len=10, start=40, mode='impact', focus='fade')
view

By default each feature's **full** impact is painted on every residue it spans (app-fidelity colouring). The per-residue values are on `view.dict_impact`; pass `normalize_by_span=True` for the span-normalized sum used by the `CPPPlot.feature_map` top bar.

In [4]:
df_imp = pd.DataFrame({'residue': list(view.dict_impact),
                       'impact': [round(v, 3) for v in view.dict_impact.values()]})
aa.display_df(df_imp, n_rows=10, show_shape=True)

DataFrame shape: (30, 2)


,residue,impact
1,40,1.200000
2,41,1.200000
3,42,1.200000
4,43,1.200000
5,44,1.200000
6,45,1.200000
7,46,1.200000
8,47,1.200000
9,48,1.200000
10,49,1.200000


## Zoom to the window, highlight residues, and pLDDT mode

`focus="zoom"` points the camera at the window (override it with `focus_region`); `highlight` paints one or more `(start, stop)` residue ranges in bright cyan (here `(42, 45)`), the same colour `AAPredPlot.predict_sample` shades on the sequence viewer, so a selection there mirrors here. `mode="plddt"` paints the AlphaFold confidence palette (blue = high, orange = low) instead of the impact ramp.

In [5]:
view_zoom = cpps_plot.map_structure(df_feat=df_feat, uniprot='P61626', tmd_len=10, start=40,
                              mode='impact', focus='zoom', focus_region=(40, 49),
                              highlight=(42, 45))
view_zoom

In [6]:
view_plddt = cpps_plot.map_structure(df_feat=df_feat, uniprot='P61626', tmd_len=10, start=40,
                               mode='plddt', focus='whole')
view_plddt

Export the interactive view as a self-contained page with `view.write_html('structure.html')`. For the structure next to the feature map use `CPPStructurePlot.plot_combined`; for a live, re-predicting explorer use `CPPStructurePlot.interactive`.